In [1]:
from model.Model import model_MSG
# from model.Plotter import plotter_MSG
# from model.coders_linkage import CODERS_pull
from pstats import SortKey, Stats

import os
import sys
import pandas as pd
# Autoreload modules when changes are applied to them
%load_ext autoreload 
%autoreload all


In [2]:

# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

%reload_ext autoreload
# Set-up
# Sys and OS Operations
os.chdir("../../")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
msg = model_MSG(scenario_name='baseline',config=None)

2025-03-24 11:14:32,068  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2025-03-24 11:14:32,072  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/muhammadawais/.local/share/ixmp/localdb/default' (user: ixmp)...


In [4]:
platform, scenario, msg_folder = msg.setup_scenario()


Model Logs: Setting up scenario


		Model creation time: 2 Min and 8 Sec 		
Model Logs: Solving the model...

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 03/24/25 11:16:53 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225c/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__baseline__v0.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__baseline__v0.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_M

In [5]:
df = scenario.par('bound_emission')

In [7]:
with scenario.transact(''):
    scenario.remove_par('bound_emission', df)

In [ ]:
df = scenario.par('bound_activity_lo',filters = {'year_act':2020})

In [ ]:
df.loc[df['technology'].str.contains('extr'),'value'] *= 0.9

In [ ]:
df.loc[df['technology'].str.contains('extr'),'value']

63     106.789217
64      58.855770
65       3.889569
66       0.000000
67       0.075733
          ...    
160      0.000000
161      0.000000
162      0.000000
163      0.000000
164      0.000000
Name: value, Length: 102, dtype: float64

In [ ]:
df['value'] = 0.6

In [ ]:
with scenario.transact(''):
    scenario.add_par('bound_activity_lo',df)

In [ ]:
scenario.remove_solution()

In [ ]:
with scenario.transact(''):
    scenario.add_par('historical_activity', df)

In [ ]:
scenario.par('historical_activity').to_csv('historical_activity.csv')

In [ ]:
df = pd.read_csv('historical_activity.csv')

In [ ]:
df = scenario.par('historical_activity')

In [ ]:
nodes = [
    'Ontario',
'Alberta',
'Saskatchewan',
'NewfoundlandandLabrador',
'Quebec',
'Manitoba',
'NorthwestTerritories',
'Nunavut'
]


In [ ]:
# df.loc[df['technology'].str.contains('gas_imp'),'value'] *= 1.3
# df.loc[df['technology'].str.contains('gas_exp'),'value'] *= 0.7
df.loc[(df['technology'].str.contains('loil_imp')) & (df['node_loc'].isin(nodes)),'value'] *= 0.9

In [ ]:
scen1 = scenario.clone(model='baseline', scenario='baseline_1')

In [60]:
scen2.par('historical_activity').to_csv('hist_act.csv')

In [37]:
df = scen2.par('bound_activity_lo')

In [41]:
df.loc[(df['year_act'] == 2035) & (df['technology'].str.contains('hydro_hc')) & (df['node_loc'] == 'NewfoundlandandLabrador'),'value'] -= 0.321387

In [42]:
with scen2.transact(''):
    scen2.add_par('bound_activity_lo',df)

In [10]:
scen2 = scenario.clone(scenario.model, "bau_2025", keep_solution=False, shift_first_model_year=2025)

In [12]:
scen2.firstmodelyear

2025

In [55]:
caseName = scen2.model + '__' + scen2.scenario + '__v' + str(scen2.version)
scen2.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '2'})

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 03/24/25 12:19:53 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225c/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__bau_2025__v1.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__bau_2025__v1.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__bau_2025__v1.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
     

2025-03-24 12:20:20,373 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-03-24 12:20:20,374 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [ ]:
from model.policies.apply_renewable_potential import apply_renewable_potential

In [ ]:
apply_renewable_potential(msg)

####################################################################################################
 Added renewable potential to the model!
####################################################################################################


In [ ]:
scenario.remove_solution()

In [ ]:
from itertools import product

import pandas as pd

In [29]:
years = scenario.par('demand')['year'].unique()
regions = scenario.par('demand')['node'].unique()
changing_first_model_year = False

In [ ]:
df_hist = scenario.par('historical_activity')

In [ ]:
# Define the commodity list
commodity_list = ['hydrogen']

# Initialize an empty set to collect unique technologies
tec_set = set()

# Iterate through each commodity and extract technologies
for commodity in commodity_list:
    output_data = scenario.par("output", {"commodity": commodity})
    
    if not output_data.empty:  # Ensure the query returned data
        tec_set.update(output_data["technology"].unique())  # Add unique technologies

# Convert the set to a list
tec_list = list(tec_set)

In [ ]:
df = scenario.par('bound_activity_up')

In [ ]:
df_n = df[(df['technology'] == 'elec_exp') &(df['year_act'] == 2020)]

In [ ]:
df_n.to_csv('test.csv')


In [ ]:
df_hist = df_n

In [ ]:
# List of new technologies to add
new_technologies = [
    'h2_bio', 'h2_smr_ccs', 'h2_t_d', 'h2_coal', 'h2_elec', 'h2_atr_ccs', 
    'h2_coal_ccs', 'h2_atr', 'h2_stor_geo', 'h2_smr', 'h2_bio_ccs', 'lh2_regas'
]

# Create a new DataFrame to store additional rows
new_rows = []

# Iterate through each unique node_loc and year_act
for (node, year) in df_hist.groupby(['node_loc', 'year_act']).groups.keys():
    for tech in new_technologies:
        new_rows.append([node, tech, year, "M1", "year", 0, "GWa"])  # Assign default values (value=0, unit="GWa")

# Convert to DataFrame
df_new_techs = pd.DataFrame(new_rows, columns=['node_loc', 'technology', 'year_act', 'mode', 'time', 'value', 'unit'])

# Append to original dataframe
df_updated = pd.concat([df_hist, df_new_techs], ignore_index=True)



In [ ]:
df_updated = df_updated[df_updated['technology'] != 'elec_exp']

In [ ]:
with scenario.transact():
    scenario.add_par('bound_activity_up', df_updated)

In [ ]:
# Define the commodity list
commodity_list = ['i_feed', 'i_spec', 'i_therm', 'rc_spec', 'rc_therm', 'transport']

# Initialize an empty set to collect unique technologies
tec_set = set()

# Iterate through each commodity and extract technologies
for commodity in commodity_list:
    output_data = scenario.par("output", {"commodity": commodity})
    
    if not output_data.empty:  # Ensure the query returned data
        tec_set.update(output_data["technology"].unique())  # Add unique technologies

# Convert the set to a list
tec_list = list(tec_set)

# Print or return the technology list
print(tec_list)


['Trans_3', 'foil_i', 'loil_i', 'ethanol_fs', 'RCspec_2', 'gas_fs', 'heat_rc', 'sp_eth_I', 'foil_rc', 'hp_gas_i', 'Trans_2', 'RCtherm_2', 'Feeds_1', 'Feeds_5', 'sp_el_RC', 'elec_trp', 'Itherm_5', 'coal_rc', 'loil_fs', 'biomass_i', 'meth_i', 'meth_rc', 'Itherm_4', 'h2_rc', 'solar_i', 'hp_gas_rc', 'Feeds_3', 'Itherm_1', 'Feeds_4', 'eth_ic_trp', 'Trans_4', 'RCspec_1', 'sp_el_I', 'RCspec_3', 'gas_i', 'Trans_1', 'eth_fc_trp', 'Ispec_2', 'solar_rc', 'Itherm_3', 'biomass_rc', 'coal_fs', 'h2_fc_I', 'foil_trp', 'heat_i', 'eth_i', 'sp_liq_I', 'Itherm_2', 'RCtherm_3', 'h2_fc_RC', 'h2_fc_trp', 'hp_el_rc', 'coal_i', 'h2_gas_fc_trp', 'RCtherm_5', 'Trans_5', 'h2_i', 'meth_fc_trp', 'gas_rc', 'RCspec_4', 'elec_rc', 'elec_i', 'methanol_fs', 'gas_trp', 'sp_meth_I', 'Ispec_5', 'loil_trp', 'eth_rc', 'Ispec_1', 'hp_el_i', 'Ispec_4', 'RCspec_5', 'meth_ic_trp', 'RCtherm_4', 'Ispec_3', 'Feeds_2', 'RCtherm_1', 'foil_fs', 'loil_rc']


In [ ]:
# Filter dataframe to only include technologies in tec_list
df_filtered = df_hist[df_hist['technology'].isin(tec_list)].copy()

# Ensure 'year_act' column exists and contains 2015 data
if 'year_act' in df_filtered.columns and 2015 in df_filtered['year_act'].values:
    # Create a copy of rows where year_act is 2015 and set year_act to 2020
    df_2020 = df_filtered[df_filtered['year_act'] == 2015].copy()
    df_2020['year_act'] = 2020  # Update year to 2020
    
    # Append the new 2020 rows to the original dataframe
    df_filtered = pd.concat([df_filtered, df_2020], ignore_index=True)
    
   
else:
    print("No records found for year_act 2015 in df_hist.")


In [ ]:
with scenario.transact(''):
    scenario.add_par('historical_activity', df_filtered)

In [ ]:
dem = scenario.par('demand')

In [ ]:
df = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v31.xlsx',sheet_name='demand')

In [ ]:
changing_first_model_year = False

In [27]:
def _calculate_demand(scenario, reg, years, changing_first_model_year):
    # Retrieve current demands
    df = (
        scenario.par("demand", {"node": reg})
        .drop_duplicates(["commodity", "level"])
        .set_index(["commodity", "level"])
    )

    dem = pd.DataFrame(
        columns=list(scenario.par("demand").columns), index=product(df.index, years)
    )

    dem = dem.assign(node=reg, time="year", unit="GWa").reset_index()
    dem["commodity"] = dem.apply(lambda r: r["index"][0][0], axis=1)
    dem["level"] = dem.apply(lambda r: r["index"][0][1], axis=1)
    dem["year"] = dem.apply(lambda r: int(r["index"][1]), axis=1)
    dem = dem.drop("index", axis=1)

    for i in dem.index:
        tmp = dem.loc[[i]]
        # Reading data for demand from the model
        # List of technologies feeding into this demand commodity
        tec_list = list(
            set(
                (
                    scenario.par(
                        "output",
                        {
                            "level": tmp.level,
                            "node_loc": reg,
                            "commodity": tmp.commodity,
                        },
                    )
                )["technology"]
            )
        )

        # Summing historical activity of these technologies

        if changing_first_model_year is False:
            par = "historical_activity"
        else:
            par = "bound_activity_lo"
        hist_act = sum(
            scenario.par(
                par,
                {"technology": tec_list, "year_act": tmp.year, "node_loc": reg},
            )["value"]
        )
        tmp["value"] = hist_act
        dem = tmp.combine_first(dem)
    dem = dem.reset_index(drop=True)

    #    for i in dem.index:
    #        dem.loc[[i], "commodity"] = i[0][0]
    #        dem.loc[[i], "level"] = i[0][1]
    #        dem.loc[[i], "year"] = int(i[1])
    #        dem["node"] = reg
    #        dem["time"] = "year"
    #        dem["unit"] = "GWa"
    #
    #        # Reading data for demand from the model
    #        # List of technologies feeding into this demand commodity
    #        tec_list = list(
    #            set(
    #                (
    #                    scenario.par(
    #                        "output",
    #                        {"level": i[0][1], "node_loc": reg, "commodity": i[0][0]},
    #                    )
    #                )["technology"]
    #            )
    #        )
    #
    #        # Summing historical activity of these technologies
    #
    #        if changing_first_model_year is False:
    #            hist_act = sum(
    #                scenario.par(
    #                    "historical_activity",
    #                    {"technology": tec_list, "year_act": i[1], "node_loc": reg},
    #                )["value"]
    #            )
    #            dem.loc[[i], "value"] = hist_act
    #
    #        else:
    #            hist_act = sum(
    #                scenario.par(
    #                    "bound_activity_lo",
    #                    {"technology": tec_list, "year_act": i[1], "node_loc": reg},
    #                )["value"]
    #            )
    #            dem.loc[[i], "value"] = hist_act
    #
    #    dem = dem.reset_index(drop=True)

    if not dem.empty:
        print(
            '> Parameter "demand" in "' + reg + '" was calibrated to'
            " the historical data from the model."
        )
    else:
        print('> There is no historical data in "' + reg + '" !!!')

    return dem



In [ ]:
changing_first_model_year

False

In [32]:
years = [2020, 2025, 2030, 2035, 2040, 2045, 2050]
import pandas as pd

all_dfs = []  # Initialize an empty list to store dataframes

for reg in regions:
    # Retrieve old demands
    df_old = scenario.par("demand", {"node": reg, "year": years})

    # Create df with recalculated demands
    df = _calculate_demand(scenario, reg, years, changing_first_model_year)

    df["unit"] = "GWa"
    df = df.round(2)

    # Append to the list
    all_dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(all_dfs, ignore_index=True)
rate_of_change = {
    2025: 0.052566283,
    2030: 0.049920952,
    2035: 0.045423593,
    2040: 0.044337335,
    2045: 0.040088765,
    2050: 0.038095737,
    2055: 0.038278554,
    2060: 0.03846426
}

# Ensure `final_df` has the correct structure and only contains 2020 values
df_2020 = final_df[final_df['year'] == 2020].copy()

# Set index for better manipulation
df_2020.set_index(["node", "commodity", "level"], inplace=True)

# Create projections based on rate of change
projections = []
for (node, commodity, level), row in df_2020.iterrows():
    base_value = row["value"]
    for year, rate in rate_of_change.items():
        projected_value = base_value * (1 + rate)  # Applying growth rate
        projections.append([node, commodity, level, year, "year", projected_value, row["unit"]])

# Convert projections to DataFrame with correct indexing
df_projections = pd.DataFrame(projections, columns=["node", "commodity", "level", "year", "time", "value", "unit"])

# Reset index for merging
df_2020.reset_index(inplace=True)

# Combine original and projected data
df_final = pd.concat([df_2020, df_projections]).reset_index(drop=True)



> Parameter "demand" in "NewfoundlandandLabrador" was calibrated to the historical data from the model.
> Parameter "demand" in "PrinceEdwardIsland" was calibrated to the historical data from the model.
> Parameter "demand" in "NovaScotia" was calibrated to the historical data from the model.
> Parameter "demand" in "NewBrunswick" was calibrated to the historical data from the model.
> Parameter "demand" in "Quebec" was calibrated to the historical data from the model.
> Parameter "demand" in "Ontario" was calibrated to the historical data from the model.
> Parameter "demand" in "Manitoba" was calibrated to the historical data from the model.
> Parameter "demand" in "Saskatchewan" was calibrated to the historical data from the model.
> Parameter "demand" in "Alberta" was calibrated to the historical data from the model.
> Parameter "demand" in "BritishColumbia" was calibrated to the historical data from the model.
> Parameter "demand" in "Yukon" was calibrated to the historical data fr

In [33]:
df_final

,node,commodity,level,year,time,value,unit
0,NewfoundlandandLabrador,i_feed,useful,2020,year,0.150000,GWa
1,NewfoundlandandLabrador,i_spec,useful,2020,year,0.310000,GWa
2,NewfoundlandandLabrador,i_therm,useful,2020,year,1.380000,GWa
3,NewfoundlandandLabrador,rc_spec,useful,2020,year,0.500000,GWa
4,NewfoundlandandLabrador,rc_therm,useful,2020,year,0.430000,GWa
...,...,...,...,...,...,...,...
697,Nunavut,transport,useful,2040,year,0.052217,GWa
698,Nunavut,transport,useful,2045,year,0.052004,GWa
699,Nunavut,transport,useful,2050,year,0.051905,GWa
700,Nunavut,transport,useful,2055,year,0.051914,GWa


In [ ]:
scenario.remove_solution()

In [35]:
with scen2.transact(''):
    scen2.add_par('demand', df_final)

In [ ]:
df = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/gdp/gdp_messageca_prov.csv')

In [ ]:
df['mode'] = 'P'

In [ ]:
df['unit'] = 'GWa/a'
df['time'] = 'year'

In [ ]:
df['node_loc'] = df['node_loc'].replace("Northwest Territories'", "NorthwestTerritories")

In [ ]:
df['node_loc'] = df['node_loc'].str.replace(" ", "", regex=True)

In [ ]:
df['node_loc'].unique()

array(['NewfoundlandandLabrador', 'PrinceEdwardIsland', 'NovaScotia',
       'NewBrunswick', 'Quebec', 'Ontario', 'Manitoba', 'Saskatchewan',
       'Alberta', 'BritishColumbia', 'Yukon', 'Northwest Territories',
       'Nunavut'], dtype=object)

In [ ]:
with scenario.transact(''):
    scenario.add_par('bound_activity_lo', df)
    scenario.add_par('bound_activity_up', df)

In [ ]:
df_old = scenario.par('bound_activity_lo',filters = {'technology':'GDP'})

In [ ]:
df_old

,node_loc,technology,year_act,mode,time,value,unit
0,Alberta,GDP,2020,P,year,230.555316,GWa/a
1,Alberta,GDP,2030,P,year,280.927712,GWa/a
2,Alberta,GDP,2040,P,year,339.206192,GWa/a
3,Alberta,GDP,2050,P,year,400.041731,GWa/a
4,Alberta,GDP,2060,P,year,463.676831,GWa/a
...,...,...,...,...,...,...,...
208,Yukon,GDP,2060,P,year,4.658699,GWa/a
209,Yukon,GDP,2025,P,year,2.547794,GWa/a
210,Yukon,GDP,2035,P,year,3.092371,GWa/a
211,Yukon,GDP,2045,P,year,3.699205,GWa/a


In [ ]:
df_new = scenario.par('demand')

In [ ]:
df_old = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v25.xlsx',sheet_name = 'demand')

In [ ]:
df_merged = df_old.merge(df_new, on = ['node','commodity','year','level'])

In [ ]:
df_merged.columns

Index(['node', 'commodity', 'level', 'year', 'time_x', 'value_x', 'unit_x',
       'time_y', 'value_y', 'unit_y'],
      dtype='object')

In [ ]:
df_merged['valdiff'] = df_merged['value_x'] - df_merged['value_y']

In [ ]:
df_merged.to_csv('demands_diff.csv')

In [ ]:
with scenario.transact(''):
    scenario.add_par('demand', df_final)

In [ ]:
df_final.to_csv('demand.csv')

In [ ]:
msg.remove_solution()

In [ ]:
df = msg.par('relation_activity')

In [ ]:
df = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v19.xlsx',sheet_name = 'relation_activity')

In [ ]:
list = ['TCE_Emission', 'TCE_Emission_NAM_PAO_EEU_WEU',
       'TCE_Emission_NAM_PAO_EEU_WEU_FSU', 'TCF4_Emission',
       'TCH4_Emission', 'THFC_Emission', 'global_GCO2', 'TN2O_Emission','TSF6_Emission']

In [ ]:
df_rel = df[df['relation'].isin(list)]

In [ ]:
df_rel_add = df[~df['relation'].isin(list)]

In [ ]:
df_rel_add

,relation,node_rel,year_rel,node_loc,technology,year_act,mode,value,unit
495,elec_relations2_renewable,Alberta,2010,Alberta,bio_istig,2010,M1,1.000000,GWa/a
496,elec_relations2_renewable,Alberta,2015,Alberta,bio_istig,2015,M1,1.000000,GWa/a
497,elec_relations2_renewable,Alberta,2020,Alberta,bio_istig,2020,M1,1.000000,GWa/a
498,elec_relations2_renewable,Alberta,2025,Alberta,bio_istig,2025,M1,1.000000,GWa/a
499,elec_relations2_renewable,Alberta,2030,Alberta,bio_istig,2030,M1,1.000000,GWa/a
...,...,...,...,...,...,...,...,...,...
284467,res_marg,Yukon,2040,Yukon,elec_t_d,2040,M1,-2.513060,GWa
284468,res_marg,Yukon,2045,Yukon,elec_t_d,2045,M1,-2.511947,GWa
284469,res_marg,Yukon,2050,Yukon,elec_t_d,2050,M1,-2.514295,GWa
284470,res_marg,Yukon,2055,Yukon,elec_t_d,2055,M1,-2.516617,GWa


In [ ]:
df_rel['node_rel'] = 'Canada'

In [ ]:
df = msg.par('relation_activity')

In [ ]:
rels_remove = ['TCE_Emission_NAM_PAO_EEU_WEU','TCE_Emission_NAM_PAO_EEU_WEU_FSU','CO2_Emission_Global_Total','TCE_regional']

In [ ]:
from model.utlities.utilities_main.message_utils import add_bound_from_hist

In [ ]:
'NewfoundlandandLabrador',''

In [ ]:

nodes = ['NovaScotia','PrinceEdwardIsland','NewBrunswick']

In [ ]:
df = scenario.par('bound_activity_lo')

In [ ]:
years = [2050,2055,2060]

In [ ]:
df.loc[
    (df['node_loc'] == 'NewfoundlandandLabrador') &
    (df['technology'].str.contains('oil_exp', na=False)) &
    (df['year_act'].isin(years)),
    'value'
] -= 4


In [ ]:
with scenario.transact('add bounds'):
    scenario.add_par('bound_activity_lo', df)

In [ ]:
df.loc[df['node_loc'].isin(nodes) & df['technology'].str.contains('oil_exp')]

,node_loc,technology,year_act,mode,time,value,unit
703,NewBrunswick,oil_exp,2020,M1,year,0.0,GWa
707,NovaScotia,oil_exp,2020,M1,year,0.0,GWa
711,PrinceEdwardIsland,oil_exp,2020,M1,year,0.0,GWa
725,NewBrunswick,oil_exp,2025,M1,year,0.0,GWa
729,NovaScotia,oil_exp,2025,M1,year,0.0,GWa
733,PrinceEdwardIsland,oil_exp,2025,M1,year,0.0,GWa
747,NewBrunswick,oil_exp,2030,M1,year,0.0,GWa
751,NovaScotia,oil_exp,2030,M1,year,0.0,GWa
755,PrinceEdwardIsland,oil_exp,2030,M1,year,0.0,GWa
769,NewBrunswick,oil_exp,2035,M1,year,0.0,GWa


In [ ]:
bound_lo_yr = 0

In [ ]:
yrs = [int(x) for x in msg.set("year").tolist() if int(x) >= msg.firstmodelyear]
yr_hist = max([x for x in msg.set("year") if x < msg.firstmodelyear])

# Loading historical data
df_hist = msg.par(
    "historical_activity",
    {"node_loc": nodes, "technology": tecs, "year_act": yr_hist},
)

In [ ]:
msg.check_out()

In [ ]:
df_bdl = pd.DataFrame(columns=msg.par("bound_activity_lo").columns)
for yr in yrs:
    d = df_hist.copy()
    d["year_act"] = yr
    d["value"] *= 1 + (yr - yr_hist) * bound_lo_yr
    df_bdl = pd.concat([df_bdl, d], ignore_index=True)
df_bdl.loc[df_bdl["value"] < 0, "value"] = 0
msg.add_par("bound_activity_lo", df_bdl)
print(
    '- Lower bound with grwoth of "{}" per year addedr'
    ' in "{}".'.format(bound_lo_yr, nodes)
)


- Lower bound with grwoth of "0" per year addedr in "['Alberta', 'BritishColumbia', 'Manitoba', 'NewBrunswick', 'NewfoundlandandLabrador', 'NorthwestTerritories', 'NovaScotia', 'Nunavut', 'Ontario', 'PrinceEdwardIsland', 'Quebec', 'Saskatchewan', 'Yukon']".


In [ ]:
remove_og_bound(msg)

In [ ]:
df_glo = msg.par('growth_activity_lo')
df_gup = msg.par('growth_activity_up')
df_iup = msg.par('initial_activity_up')
df_ilow = msg.par('initial_activity_lo')


import pandas as pd

# Define the filter function
def remove_extraction_rows(df):
    return df[df.apply(lambda row: row.astype(str).str.contains('gas_exp|oil_exp', regex=True)).any(axis=1)]

# Apply the function to each dataframe
df_glo = remove_extraction_rows(df_glo)
df_gup = remove_extraction_rows(df_gup)
df_iup = remove_extraction_rows(df_iup)
df_ilow = remove_extraction_rows(df_ilow)


In [ ]:
with msg.transact(''):
    msg.remove_par('growth_activity_lo', df_glo)
    msg.remove_par('growth_activity_up', df_gup)
    msg.remove_par('initial_activity_up', df_iup)
    msg.remove_par('initial_activity_lo', df_ilow)

In [ ]:
nodes = msg.set('node').tolist()[2:]


In [ ]:
add_bound_from_hist(msg,tecs, nodes, bound_lo_yr = 0.0001)

>>>Warning<<<: No historical activity for technologies "['Alberta', 'BritishColumbia', 'Manitoba', 'NewBrunswick', 'NewfoundlandandLabrador', 'NorthwestTerritories', 'NovaScotia', 'Nunavut', 'Ontario', 'PrinceEdwardIsland', 'Quebec', 'Saskatchewan', 'Yukon']" in "['gas_exp', 'oil_exp']", no bounds added.


In [ ]:
df = msg.par('bound_actio')

In [ ]:
msg.remove_solution()

In [ ]:
transport_update(msg)

- coal_trp removed.
- Capex removed and vintaging corrected for ['h2_fc_trp']
- O&M updated for transport technologies.


In [ ]:
df = msg.set('cat_emission')

In [ ]:
df_remove = df[~df['emission'].isin(emis)]

In [ ]:
import pandas as pd
import pandas as pd

# Define the full dataset exactly as provided
data = ["Ontario", "gas_exp", 2020, "M1", "year", 4.65615e-12]
    



# Convert into DataFrame
df_condition = pd.DataFrame(data, columns=['node_loc', 'technology', 'year_act','mode' ,'time', 'value'])

# Load another target dataset from which values will be subtracted (Example data)
data_target = msg.par('bound_activity_lo',filters = {'technology': ['gas_exp','oil_exp']})

df_target = data_target

#
# Filter df_target based on matching 'node_loc' and 'year' from df_condition
df_target_filtered = df_target[
    df_target[['node_loc', 'year_act']].apply(tuple, axis=1).isin(
        df_condition[['node_loc', 'year_act']].apply(tuple, axis=1)
    )
]

In [ ]:

# Load another target dataset from which values will be subtracted (Example data)
data_target = msg.par('bound_activity_lo',filters = {'technology': ['gas_exp']})

In [ ]:
data_target.loc[(data_target['node_loc'] == 'Ontario') & (data_target['year_act'] == 2020), 'value'] -=4.65615e-12

In [ ]:
with msg.transact():
    msg.add_par('bound_activity_lo', data_target)

In [ ]:
# Merge the filtered target dataset with the condition dataset
df_merged = df_target_filtered.merge(df_condition, on=['node_loc', 'technology', 'year_act', 'mode', 'time'], how='inner', suffixes=('_target', '_condition'))


In [ ]:

# Perform the subtraction
df_merged['value'] = df_merged['value_target'] - df_merged['value_condition']

# Keep only relevant columns
df_final = df_merged[['node_loc', 'technology', 'year_act', 'mode', 'time', 'value']]



In [ ]:
with msg.transact():
    msg.add_par('bound_activity_lo', df_final)

In [ ]:
emis = ['TCE','TCE_CO2', 'TCE_non-CO2']

In [ ]:
df_emis_maintain = df[df['emission'].isin(emis)]

In [ ]:
df_emis_remove = df[~df['emission'].isin(emis)]

In [ ]:
df = df[df['relation'].isin(['CO2_Emission_Global_Total'])]

df_Reob

In [ ]:
import pandas as pd

# Use an alternative way to convert range to list
type_year_values = [year for year in range(2025, 2061, 5)]

# Manually create the DataFrame
df_bound_net = pd.DataFrame({
    "node": ["Canada"] * len(type_year_values),
    "type_emission": ["TCE"] * len(type_year_values),
    "type_tec": ["all"] * len(type_year_values),
    "type_year": type_year_values,
    "value": [200] * len(type_year_values)
})




     node type_emission type_tec  type_year  value
0  Canada           TCE      all       2025    200
1  Canada           TCE      all       2030    200
2  Canada           TCE      all       2035    200
3  Canada           TCE      all       2040    200
4  Canada           TCE      all       2045    200
5  Canada           TCE      all       2050    200
6  Canada           TCE      all       2055    200
7  Canada           TCE      all       2060    200


In [ ]:
msg.remove_solution()

In [ ]:
with msg.transact(''):
    # msg.remove_par('bound_emission', df)
    msg.add_par('bound_emission', df_upd)

In [ ]:
df = msg.par('bound_emission')

In [ ]:

from model.utlities.demands import adjust_demands

In [ ]:
df = msg.par('demand')


In [ ]:
import pandas as pd

# Define cumulative rate of change
rate_of_change = {
    2025: 0.052566283,
    2030: 0.049920952,
    2035: 0.045423593,
    2040: 0.044337335,
    2045: 0.040088765,
    2050: 0.038095737,
    2055: 0.038278554,
    2060: 0.03846426
}

# Convert year column to integer
df["year"] = df["year"].astype(int)

# Get the 2020 base values for each (node, commodity) pair
base_values = df[df["year"] == 2020].set_index(["node", "commodity"])["value"]

# Sort DataFrame to ensure chronological updates
df = df.sort_values(by=["node", "commodity", "year"])

# Apply cumulative rate of change iteratively
for year in sorted(rate_of_change.keys()):
    mask = df["year"] == year
    previous_year = year - 5  # Assuming a 5-year interval between records
    prev_mask = df["year"] == previous_year

    df.loc[mask, "value"] = df.loc[mask].apply(
        lambda row: df.loc[
            (df["node"] == row["node"]) & (df["commodity"] == row["commodity"]) & (df["year"] == previous_year), "value"
        ].values[0] * (1 + rate_of_change[year])
        if not df.loc[
            (df["node"] == row["node"]) & (df["commodity"] == row["commodity"]) & (df["year"] == previous_year), "value"
        ].empty else base_values.get((row["node"], row["commodity"]), 0) * (1 + rate_of_change[year]),
        axis=1
    )


In [ ]:
with msg.transact(''):
    msg.add_par('demand', df)

In [ ]:
(5.715158- 5.443418)/5.443418

0.049920840177991

In [ ]:
df_upd = df[df['type_year'] != 'cumulative']

In [ ]:
df_upd['value'] = 300

In [ ]:
df['value'] = df['value'] 

In [ ]:
df = msg.par('bound_emission')

In [ ]:
df.loc[df['type_year']== '2020','value'] = 350

In [ ]:
df['type_year'] ='2020'

In [ ]:
scenario.par('historical_activity')['technology'].unique()

array(['bio_ppl', 'biomass_i', 'biomass_nc', 'biomass_rc', 'biomass_t_d',
       'coal_fs', 'coal_i', 'coal_ppl', 'coal_rc', 'coal_t_d', 'elec_trp',
       'eth_bio', 'eth_ic_trp', 'foil_fs', 'foil_i', 'foil_ppl',
       'foil_rc', 'foil_t_d', 'foil_trp', 'gas_cc', 'gas_ct', 'gas_fs',
       'gas_i', 'gas_ppl', 'gas_rc', 'gas_t_d', 'gas_trp', 'heat_i',
       'heat_rc', 'heat_t_d', 'land_use_biomass', 'landfill_ele',
       'loil_fs', 'loil_i', 'loil_ppl', 'loil_rc', 'loil_t_d', 'loil_trp',
       'ref_hil', 'ref_lol', 'solar_pv_ppl', 'solar_rc', 'sp_el_I',
       'sp_el_RC', 'wind_ppl', 'elec_i', 'elec_t_d', 'hydro_hc',
       'hydro_lc', 'coal_extr_mpen', 'oil_extr_mpen', 'gas_extr_mpen',
       'elec_exp', 'elec_imp', 'coal_imp', 'coal_exp', 'oil_imp',
       'oil_exp', 'gas_exp', 'gas_imp', 'lignite_extr', 'gas_t_d_ch4',
       'loil_imp', 'coal_extr_ch4', 'oil_extr_1', 'gas_extr_1',
       'oil_extr_2', 'gas_extr_2', 'oil_extr_3', 'gas_extr_3',
       'oil_extr_4', 'gas_extr_4', '

In [ ]:
['bio_ppl','coal_adv_ccs', 'nuc_lc','solar_pv_ppl','hydro_hc','hydro_lc','loil_ppl','gas_cc', 'gas_ct','gas_ppl','eth_bio'

In [ ]:
com = 'secondary'
node = 'Alberta'
yr = 2020

In [ ]:
df = scenario.par('output', {'level': com, 'node_loc': node})

In [ ]:
tecs = df['technology'].unique()

In [ ]:
tecs

array(['GDP', 'Population', 'solar_pv_ppl', 'wind_ppl', 'nuc_lc',
       'hydro_hc', 'hydro_lc', 'gas_exp', 'oil_exp'], dtype=object)

In [ ]:
df = scenario.par('bound_activity_lo')

In [ ]:


tecs = ['solar_pv_ppl', 'wind_ppl', 'nuc_lc',
       'hydro_hc', 'hydro_lc', 'gas_exp', 'oil_exp']

In [ ]:
df = pd.read_excel('data/filtered_fix_cost_no_obps.xlsx')

In [ ]:
convert_loss_to_activity(scenario,tecs)

In [31]:
import pandas as pd
from itertools import product

In [ ]:
import pandas as pd
from itertools import product

def convert_loss_to_activity(msg, tec_list, node_list=[]):
    """
    Updating historical activity of a technology by calculating it as a loss
    (the existing value is considered a loss, and will be converted to a new
    value as activity).

    Parameters
    ----------
    msg : message_ix.Scenario
    tec_list : list
        List of technologies whose historical activity must be converted as a loss
        between their input and output.
    node_list : list, optional
        List of model nodes to which these changes must be applied. The default is [].
    """
    # List of nodes for applying the calibration in this module
    if not node_list:
        node_list = [x for x in msg.set("node") if x not in ["World"]]

    parname = "bound_activity_lo"
    msg.check_out()

    # Finding historical years
    yr_hist = [x for x in set(msg.set("year")) if x < msg.firstmodelyear]

    for tec, node in product(tec_list, node_list):
        df = msg.par(parname, {"technology": tec, "node_loc": node}).set_index("year_act")

        # Fetch input data
        inp_df = msg.par("input", {"technology": tec, "node_loc": node, "year_act": yr_hist})
        out_df = msg.par("output", {"technology": tec, "node_loc": node, "year_act": yr_hist})

        # Ensure "value" column is numeric and handle potential string concatenation issues
        inp_df["value"] = pd.to_numeric(inp_df["value"], errors="coerce")
        out_df["value"] = pd.to_numeric(out_df["value"], errors="coerce")

        # Drop rows where conversion fails (i.e., non-numeric values)
        inp_df = inp_df.dropna(subset=["value"])
        out_df = out_df.dropna(subset=["value"])

        # Group and calculate mean values
        inp = inp_df.groupby("year_act")["value"].mean()
        out = out_df.groupby("year_act")["value"].mean()

        # Calculate activity based on "input", "output", and loss
        for yr in df.index:
            if yr in inp and yr in out and inp[yr] != 0:
                df.loc[yr, "value"] = df["value"][yr] /  inp[yr]

        # Add data to the scenario and commit
        msg.add_par(parname, df.reset_index())

    msg.commit("Historical data recalibrated based on the loss to activity.")


In [ ]:
df = scenario.set('cat_year')

In [ ]:
node_commodity=[("NovaScotia", "useful", "rc_spec", 2020)]
for node, lvl, com, yr in node_commodity:
    # # Demand
    dem = scenario.par(
        "demand", {"commodity": com, "node": node, "level": lvl, "year": yr}
    )
    dem = float(dem["value"])

    # Activity
    out = scenario.par("output", {"commodity": com, "node_loc": node, "year_act": yr})
    tec_list = list(set(out["technology"]))
    act = scenario.par(
        "historical_activity", {"technology": tec_list, "node_loc": node, "year_act": 2015}
    )["value"].sum()


In [ ]:
act

0.5690984436998591

In [ ]:
dem

0.57

In [ ]:
df = scenario.par('bound_activity_lo')

In [ ]:
df = df[df['technology'].str.contains('exp')]

In [ ]:
with scenario.transact(''):
    scenario.remove_par('bound_activity_lo', df)

In [ ]:
dem

1.96

In [ ]:
historical_activity = scenario.par('historical_activity')

In [ ]:
# Define the provinces involved in the analysis
PROVINCES = [
    "Alberta",
    "BritishColumbia",
    "Manitoba",
    "Ontario",
    "NewBrunswick",
    "NewfoundlandandLabrador",
    "NovaScotia",
    "PrinceEdwardIsland",
    "Quebec",
    "Saskatchewan",
    "Yukon",
]

# Define the technologies associated with each energy type
foil_techs = [
    "foil_fs",
    "foil_i",
    "foil_ppl",
    "foil_rc",
    "foil_trp",
]
loil_techs = [
    "loil_fs",
    "loil_i",
    "loil_ppl",
    "loil_rc",
    "loil_trp",
]
gas_techs = [
    "gas_cc",
    "gas_ct",
    "gas_fs",
    "gas_i",
    "gas_ppl",
    "gas_rc",
    "gas_trp",
]
coal_techs = [
    "coal_fs",
    "coal_i",
    "coal_ppl",
    "coal_rc",
    "coal_trp",
]

def sum_demand(df: pd.DataFrame, techs: list):
    """
    Sum the demand for a given set of technologies within the DataFrame.

    Args:
                    df (pd.DataFrame): DataFrame containing historical activity data.
                    techs (list): List of technology names to sum demand for.

    Returns:
                    pd.Series: Series with the summed demand for each province and year.
    """
    return (
        df[df["technology"].isin(techs)]
        .groupby(["year_act", "node_loc"])["value"]
        .sum()
    )

def balance_energy(
    row: pd.Series, prod_col: str, imp_col: str, exp_col: str, demand_col: str
):
    """
    Balance the energy production, import, and export for a specific energy type.

    Args:
                    row (pd.Series): Row of data containing production, import, export, and demand values.
                    prod_col (str): Column name for production data.
                    imp_col (str): Column name for import data.
                    exp_col (str): Column name for export data.
                    demand_col (str): Column name for demand data.

    Returns:
                    pd.Series: Updated row with balanced production, import, and export values.
    """
    production = row[prod_col]
    imports = row[imp_col]
    exports = row[exp_col]
    demand = row[demand_col]

    # Calculate the net balance of energy
    net_balance = production + imports - exports

    # Adjust imports or exports to meet the demand
    if net_balance < demand:
        imports += demand - net_balance
    elif net_balance > demand:
        exports += net_balance - demand

    # Ensure imports are not negative
    if imports < 0:
        exports += abs(imports)
        imports = 0

    # Update the row with the balanced values
    row[imp_col] = imports
    row[exp_col] = exports
    return row

In [ ]:
balanced_df = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/oil_gas_production/commodity_balance_2020.csv')

In [ ]:
# Get the unique years in the historical activity data
years = [2015]

# Iterate over each year to balance energy production, imports, and exports
for year in years:
    # Filter data for the current year and relevant provinces
    df_year = historical_activity[
        (historical_activity["year_act"] == year)
        & (historical_activity["node_loc"].isin(PROVINCES))
    ]

    # Calculate the total demand for each energy type
    coal_demand = sum_demand(df_year, coal_techs)
    foil_demand = sum_demand(df_year, foil_techs)
    loil_demand = sum_demand(df_year, loil_techs)
    gas_demand = sum_demand(df_year, gas_techs)
    oil_demand = foil_demand.add(
        loil_demand, fill_value=0
    )  # Combine demands for foil and loil

    # Create a pivot table to summarize the production, import, and export data
    pivot_df = df_year.pivot_table(
        index="node_loc", columns="technology", values="value", aggfunc="sum"
    ).reset_index()

    # Define the required columns for production, import, and export data
    required_columns = [
        "coal_extr_mpen",
        "coal_imp",
        "coal_exp",
        "gas_extr_mpen",
        "gas_imp",
        "gas_exp",
        "oil_extr_mpen",
        "oil_imp",
        "oil_exp",
    ]

    # Ensure all required columns are present in the pivot table
    for tech in required_columns:
        if tech not in pivot_df.columns:
            pivot_df[tech] = 0

    # Extract the production, import, and export data for the current year and provinces
    production_import_export_data = pivot_df[
        ["node_loc"] + required_columns
    ].fillna(0)

    # Create a DataFrame to hold the demand data for the current year and provinces
    demand_data = pd.DataFrame(
        {
            "node_loc": coal_demand.index.get_level_values("node_loc"),
            "coal_demand": coal_demand.values,
            "gas_demand": gas_demand.values,
            "oil_demand": oil_demand.values,
        }
    ).fillna(0)

    # Merge the production/import/export data with the demand data
    merged_df = production_import_export_data.merge(
        demand_data, on="node_loc", how="left"
    ).fillna(0)

    # Create a copy of the merged data to apply balancing
    balanced_df = merged_df.copy()

    # Apply the balance_energy function to coal, gas, and oil data
    balanced_df = balanced_df.apply(
        balance_energy,
        axis=1,
        prod_col="coal_extr_mpen",
        imp_col="coal_imp",
        exp_col="coal_exp",
        demand_col="coal_demand",
    )

    balanced_df = balanced_df.apply(
        balance_energy,
        axis=1,
        prod_col="gas_extr_mpen",
        imp_col="gas_imp",
        exp_col="gas_exp",
        demand_col="gas_demand",
    )

    balanced_df = balanced_df.apply(
        balance_energy,
        axis=1,
        prod_col="oil_extr_mpen",
        imp_col="oil_imp",
        exp_col="oil_exp",
        demand_col="oil_demand",
    )

    # Update the historical_activity DataFrame with the balanced import and export data
    for index, row in balanced_df.iterrows():
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "coal_imp"),
            "value",
        ] = row["coal_imp"]
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "coal_exp"),
            "value",
        ] = row["coal_exp"]
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "gas_imp"),
            "value",
        ] = row["gas_imp"]
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "gas_exp"),
            "value",
        ] = row["gas_exp"]
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "oil_imp"),
            "value",
        ] = row["oil_imp"]
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "oil_exp"),
            "value",
        ] = row["oil_exp"]

# Return the updated histo


In [ ]:
balanced_df = pd.read_csv('energy_balanced.csv')

In [ ]:
year = 2020

In [ ]:
for index, row in balanced_df.iterrows():
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "coal_imp"),
            "value",
        ] = row["coal_imp"]
        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "coal_exp"),
            "value",
        ] = row["coal_exp"]

        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "coal_extr"),
            "value",
        ] = row["coal_extr_mpen"]

        historical_activity.loc[
            (historical_activity["year_act"] == year)
            & (historical_activity["node_loc"] == row["node_loc"])
            & (historical_activity["technology"] == "coal_extr_mpen"),
            "value",
        ] = row["coal_extr_mpen"]
        

In [ ]:
with scenario.transact(''):
    scenario.add_par('historical_activity',historical_activity)

In [ ]:
balanced_df.to_csv('energy_balanced_2015.csv')

In [ ]:
df = historical_activity[historical_activity['technology'] == 'oil_imp']

In [ ]:
nodes = scenario.set('node')[2:]

In [ ]:
from model.utlities.utilities_main.message_utils import add_vintage, add_bound_from_hist

In [22]:
from model.utlities.update_calibration import model_calibrate

In [23]:
nodes = list(scen2.set('node')[2:])

In [24]:
model_calibrate(scen2, nodes)

>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "Alberta"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "BritishColumbia"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "Manitoba"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "NewBrunswick"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "NewfoundlandandLabrador"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "NorthwestTerritories"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "NovaScotia"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "Nunavut"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "Ontario"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "PrinceEdwardIsland"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "Quebec"
>>>Warnin

In [ ]:
scenario.par('bound_activity_lo', filters = {'technology': ['gas_exp','oil_exp']})

,node_loc,technology,year_act,mode,time,value,unit
696,Alberta,gas_exp,2020,M1,year,8.025658e+01,GWa
697,Alberta,oil_exp,2020,M1,year,1.361884e+02,GWa
698,BritishColumbia,gas_exp,2020,M1,year,6.111775e+01,GWa
699,BritishColumbia,oil_exp,2020,M1,year,4.800240e-09,GWa
700,Manitoba,gas_exp,2020,M1,year,0.000000e+00,GWa
...,...,...,...,...,...,...,...
889,Quebec,oil_exp,2060,M1,year,0.000000e+00,GWa
890,Saskatchewan,gas_exp,2060,M1,year,0.000000e+00,GWa
891,Saskatchewan,oil_exp,2060,M1,year,2.294210e+01,GWa
892,Yukon,gas_exp,2060,M1,year,0.000000e+00,GWa


In [ ]:
add_bound_from_hist(scenario,nodeName = nodes, tecs = ['gas_exp','oil_exp',],bound_up_yr=0, bound_lo_yr=0.00001, yearly=False)

- Lower bound with grwoth of "1e-05" per year addedr in "['Alberta', 'BritishColumbia', 'Manitoba', 'NewBrunswick', 'NewfoundlandandLabrador', 'NorthwestTerritories', 'NovaScotia', 'Nunavut', 'Ontario', 'PrinceEdwardIsland', 'Quebec', 'Saskatchewan', 'Yukon']".


In [ ]:
nodes

['NovaScotia', 'PrinceEdwardIsland', 'NewBrunswick']

In [ ]:
df.to_csv('historical_activity.csv')

In [ ]:
df = pd.read_csv('historical_activity.csv')

In [ ]:
historical_activity.dropna(inplace = True)

In [ ]:
with scenario.transact():
    scenario.add_par('historical_activity', historical_activity)

In [ ]:
balanced_df.to_csv('energy_balanced_2015.csv')

In [ ]:
balanced_df

,node_loc,coal_extr_mpen,coal_imp,coal_exp,gas_extr_mpen,gas_imp,gas_exp,oil_extr_mpen,oil_imp,oil_exp,coal_demand,gas_demand,oil_demand
0,Alberta,7.924278,10.055441,16.340840,118.654686,16.559732,85.891506,282.950745,19.498195,2.455264e+02,1.638879,49.322911,56.922569
1,BritishColumbia,23.395486,0.000000,23.362482,65.395300,14.452417,68.097270,7.255265,18.662141,4.800000e-09,0.033005,11.750448,25.917407
2,Manitoba,0.000000,0.595262,0.580989,0.000000,3.804374,0.293803,3.120218,6.865578,1.471447e+00,0.014273,3.510571,8.514349
3,NewBrunswick,0.000000,2.690500,2.457440,0.057243,1.726017,0.779095,0.000000,8.136257,1.920000e-05,0.233060,1.004165,8.136238
4,NewfoundlandandLabrador,0.000000,0.072298,0.052267,0.000000,0.260084,0.000000,18.559546,2.583940,1.477050e+01,0.020031,0.260084,6.372984
5,NovaScotia,0.000000,5.732289,5.099936,0.000000,0.636396,0.000000,0.091112,8.226925,9.111220e-02,0.632352,0.636396,8.226925
6,Ontario,0.000000,3.272136,2.855145,0.084147,50.804461,4.139297,0.046995,76.147469,1.000954e-02,0.416991,46.749312,76.184454
7,PrinceEdwardIsland,0.000000,0.004315,0.000000,0.000000,0.004204,0.000000,0.000000,1.107452,1.000000e-09,0.004315,0.004204,1.107452
8,Quebec,0.000000,0.918316,0.779968,0.000000,9.341038,2.109120,0.000000,43.429961,0.000000e+00,0.138348,7.231918,43.429961
9,Saskatchewan,6.414891,3.591644,9.102017,4.321743,21.131930,15.491395,36.244127,7.037104,2.897971e+01,0.904518,9.962277,14.301525


In [ ]:
df = scenario.par('historical_activity')

In [ ]:
df = df[df['technology'].isin(['ref_lol','ref_hil'])]

In [ ]:
with scenario.transact(''):
    scenario.remove_par('historical_activity', df)

In [ ]:
df_new = df[df['node_loc'].isin(['Alberta','BritishColumbia','Saskatchewan','Ontario','Quebec','NewBrunswick'])]

In [ ]:
df = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/data/oil_gas_production/MESSAGE_Imports_Exports.xlsx')

In [ ]:
df_new['technology'].unique()

array(['oil_imp', 'oil_exp'], dtype=object)

In [ ]:
df_new = df[df['technology'].isin(['oil_imp','oil_exp'])]

In [ ]:
df_new['node_loc'] = df_new['node_loc'].str.replace(" ", "", regex=True)

In [ ]:
df_new.rename(columns = {'year_vtg':'year_act'}, inplace = True)

In [ ]:
df_new['time'] = 'year'
df_new['mode'] = 'M1'


In [ ]:
df_new

,node_loc,technology,year_act,value,unit,Unnamed: 5,Legend,Technology Type,time,mode
52,Alberta,oil_imp,2020,0.000000,Gwa,NaN,NaN,NaN,year,M1
53,Alberta,oil_exp,2020,242.086313,Gwa,NaN,NaN,NaN,year,M1
54,BritishColumbia,oil_imp,2020,0.000000,Gwa,NaN,NaN,NaN,year,M1
55,BritishColumbia,oil_exp,2020,0.000000,Gwa,NaN,NaN,NaN,year,M1
56,Saskatchewan,oil_imp,2020,0.000000,Gwa,NaN,NaN,NaN,year,M1
...,...,...,...,...,...,...,...,...,...,...
176,Yukon,oil_exp,2020,0.000000,NaN,NaN,NaN,NaN,year,M1
177,NorthwesternTerritories,oil_imp,2020,0.000000,NaN,NaN,NaN,NaN,year,M1
178,NorthwesternTerritories,oil_exp,2020,0.000000,NaN,NaN,NaN,NaN,year,M1
179,Nunavut,oil_imp,2020,0.000000,NaN,NaN,NaN,NaN,year,M1


In [ ]:
df_new['unit'] = 'GWa'

In [ ]:
df_new['node_loc'] = df_new['node_loc'].replace("NorthwesternTerritories", "NorthwestTerritories")

In [ ]:
with scenario.transact(''):
    scenario.add_par('historical_activity', df_new)

In [ ]:
df = scenario.par('historical_activity')

In [56]:
scen2.to_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v32.xlsx')

In [45]:
df_lo = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v31.xlsx', sheet_name='bound_emission')

In [49]:
df_lo = df_lo[df_lo['type_year'] != 2020]

In [51]:
df_lo['value'] = 180

In [53]:
scen2.remove_solution()

In [54]:
with scen2.transact(''):
    scen2.add_par('bound_emission', df_lo)

In [ ]:
df_up = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v19.xlsx', sheet_name='bound_activity_up')

In [ ]:
df_lo = df_lo[df_lo['technology'] == 'GDP']
df_up = df_up[df_up['technology'] == 'GDP']

In [ ]:
df_gdp_lo = scenario.par('bound_activity_lo', filters = {'technology': 'GDP'})

In [ ]:
df_gdp_up = scenario.par('bound_activity_up', filters = {'technology': 'GDP'})

In [ ]:
df_gdp_r = df_gdp[df_gdp['year_act'] == 2015]

In [ ]:
with scenario.transact('add GDP'):
    scenario.add_par('bound_activity_lo', df_lo)
    scenario.add_par('bound_activity_up', df_up)

In [ ]:
df = scenario.par('historical_activity')

In [ ]:
df_c = df[df['technology'].str.contains('coal')]

In [ ]:
df.to_excel('hist_act_c.xlsx')

In [ ]:
df['year_act'] = 2015

In [ ]:
 #Given net-zero emission trajectory from Environment Canada
data = {
    "type_year": list(range(2025, 2061, 5)),
    "value": [659, 549, 439, 329, 220, 110, 110,50,50]
}

# Create DataFrame
df_bound_net = pd.DataFrame(data)

# Add requi0.0116 * 1e3 *365 /8760red columns
df_bound_net.insert(0, "node", "Canada")
df_bound_net.insert(1, "type_emission", "TCE")
df_bound_net.insert(2, "type_tec", "all")

,relation,node_rel,year_rel,node_loc,technology,year_act,mode,value,unit
0,TCE_Emission,Canada,2010,Alberta,CF4_TCE,2010,M1,2015.450,???
1,TCF4_Emission,Canada,2010,Alberta,CF4_TCE,2010,M1,2015.450,???
2,TCE_Emission,Canada,2015,Alberta,CF4_TCE,2015,M1,2015.450,???
3,TCF4_Emission,Canada,2015,Alberta,CF4_TCE,2015,M1,2015.450,???
4,TCE_Emission,Canada,2020,Alberta,CF4_TCE,2020,M1,2015.450,???
...,...,...,...,...,...,...,...,...,...
8120,global_GCO2,Canada,2040,Yukon,oil_exp,2040,M1,-0.631,GWa/a
8121,global_GCO2,Canada,2045,Yukon,oil_exp,2045,M1,-0.631,GWa/a
8122,global_GCO2,Canada,2050,Yukon,oil_exp,2050,M1,-0.631,GWa/a
8123,global_GCO2,Canada,2055,Yukon,oil_exp,2055,M1,-0.631,GWa/a


In [ ]:
def carbon_price(msg):
    """Increase the carbon price to understand level of emissions reduction."""
    
    with msg.transact():
        df = msg.par('tax_emission')
        df['type_tec'] = 'all'
        df['value'] *=  500
        msg.add_par("tax_emission", df)

def remove_og_bound(msg):
    """Increase the carbon price to understand level of emissions reduction"""
    
    with msg.transact():
        df = msg.par('bound_activity_lo')
        df_remove = df[df['technology'].str.contains('oil_extr|gas_extr')]
        msg.remove_par("bound_activity_lo", df_remove)



In [ ]:


def transfer_demands(base, scen):
    """Copies a demand from one scenario to another.

    Copies the demand ('variable') from one scenario
    to another and adds it as a 'parameter'.

    Parameters
    ----------
    base : :class:`message_ix.Scenario`
        source scenario from which demands should be copied

    scen : :class:`message_ix.Scenario`
        target scenario to which demands should be copied
    """
    print(f"Transfer data from {base.url} variable 'DEMAND'")
    print(f"to {scen.url} parameter 'demand'")

    df = base.var("DEMAND").rename(columns={"lvl": "value"}).assign(unit="GWa")
    scen.check_out()
    scen.add_par("demand", df)
    scen.commit("parameter demand updated from source variable")

    print(f"{len(df)} observations")

In [ ]:
netzero_1.par('tax_emission')

,node,type_emission,type_tec,type_year,value,unit
0,Alberta,TCE,enduse_energy_tecs,2025,165.425062,???
1,Alberta,TCE,enduse_energy_tecs,2030,285.215625,???
2,Alberta,TCE,enduse_energy_tecs,2035,387.893250,???
3,Alberta,TCE,enduse_energy_tecs,2040,387.893250,???
4,Alberta,TCE,enduse_energy_tecs,2045,387.893250,???
...,...,...,...,...,...,...
99,Yukon,TCE,enduse_energy_tecs,2040,387.893250,???
100,Yukon,TCE,enduse_energy_tecs,2045,387.893250,???
101,Yukon,TCE,enduse_energy_tecs,2050,387.893250,???
102,Yukon,TCE,enduse_energy_tecs,2055,387.893250,???


In [ ]:
carbon_price(netzero_1)


In [ ]:
netzero_1 = scenario.clone(scenario.model, "netzero_1", keep_solution=False)

In [ ]:
netzero = netzero_1.clone(netzero_1.model, "netzero", keep_solution=False)

In [ ]:
df = netzero.par('tax_emission')
df = df[df['type_tec'] == 'all']

In [ ]:
df

,node,type_emission,type_tec,type_year,value,unit
104,Alberta,TCE,all,2025,827.125312,???
105,Alberta,TCE,all,2030,1426.078125,???
106,Alberta,TCE,all,2035,1939.466250,???
107,Alberta,TCE,all,2040,1939.466250,???
108,Alberta,TCE,all,2045,1939.466250,???
...,...,...,...,...,...,...
203,Yukon,TCE,all,2040,1939.466250,???
204,Yukon,TCE,all,2045,1939.466250,???
205,Yukon,TCE,all,2050,1939.466250,???
206,Yukon,TCE,all,2055,1939.466250,???


In [ ]:
transfer_demands(netzero_1, netzero)

Transfer data from MESSAGEix-CA/netzero_1#3 variable 'DEMAND'


to MESSAGEix-CA/netzero#23 parameter 'demand'
666 observations


In [ ]:
with netzero.transact():
    netzero.remove_par('tax_emission', df)

In [ ]:
df= netzero_1.par('bound_emission')

In [ ]:
import pandas as pd

# Define new values for each year
new_values = {
    '2025': 199.523825,
    2030: 107.426983,
    2035: 71.188808,
    2040: 45.701622,
    2045: 40,
    2050: 40,
    2055: 40,
    2060: 40
}



# Update only the values that exist in new_values
for year, new_value in new_values.items():
    df.loc[df["type_year"] == year, "value"] = new_value




In [ ]:
with netzero_1.transact():
    netzero_1.add_par('bound_emission', df)

In [ ]:
def assign_cumulative_netzero(scenario):
    # Define emissions data for Net-Zero scenario
    start_year = 2025
    target_year = 2050
    end_year = 2060

    initial_emission = 720  # MtCO2e in 2025
    net_zero_emission = 0  # Target emissions in 2050

    # Assuming a linear decline from 2025 to 2050
    years = list(range(start_year, end_year + 1))
    emissions_netzero = []

    # Calculate the annual decline rate for Net-Zero
    decline_rate_netzero = (net_zero_emission - initial_emission) / (target_year - start_year)

    # Calculate emissions for each year
    for year in years:
        if year <= target_year:
            emissions_netzero.append(initial_emission + (year - start_year) * decline_rate_netzero)
        else:  # Assuming zero emissions after 2050
            emissions_netzero.append(net_zero_emission)

    # Calculate cumulative emissions for Net-Zero scenario
    cumulative_budget_netzero = sum(emissions_netzero)

    # Convert MtCO2e to MtC
    cumulative_budget_netzero_mtC = cumulative_budget_netzero * 0.2727

    # Display results
    print(f'{cumulative_budget_netzero_mtC} million tonne of Carbon for Net Zero by 2050')

    import pandas as pd

    # Recreate the original DataFrame with the given trajectory
    data = {
        "type_year": list(range(2025, 2061, 5)),
        "value": [659, 549, 439, 329, 220, 110, 110, 50]  # Original values in MtCO2
    }

    # Create DataFrame
    df_bound_net = pd.DataFrame(data)

    # Add required columns
    df_bound_net.insert(0, "node", "Canada")
    df_bound_net.insert(1, "type_emission", "TCE")
    df_bound_net.insert(2, "type_tec", "all")

    # Convert values from MtCO2 to MtC
    df_bound_net["value"] = df_bound_net["value"] * (12 / 44)

    with netzero.transact():
        
        # netzero_1.remove_par('tax_emission', df)
        # netzero_1.add_par('bound_emission', df)
        netzero.add_par('bound_emission', df_bound_net)

In [ ]:
0.0116 * 1e3 *365 /8760

0.48333333333333334

In [ ]:
860*0.48


412.8

In [ ]:
data = {
        "type_year": list(range(2025, 2061, 5)),
        "value": [659, 549, 439, 329, 220, 110, 110,50,50]
    }

In [ ]:
data

{'type_year': [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060],
 'value': [659, 549, 439, 329, 220, 110, 110, 50, 50]}

In [ ]:
df = netzero_1.par('tax_emission')

In [ ]:
df.loc[df['type_tec'] == 'all','value'] /= 500

In [ ]:
df.loc[df['type_tec'] == 'all','value'] *= 5

In [ ]:
with netzero_1.transact(''):
    netzero_1.add_par('tax_emission', df)

In [ ]:
netzero_1.remove_solution()

In [ ]:
assign_cumulative_netzero(netzero)

2552.472 million tonne of Carbon for Net Zero by 2050


In [ ]:
caseName = netzero.model + '__' + netzero.scenario + '__v' + str(netzero.version)
netzero.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '2'})

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 03/17/25 15:28:21 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225b/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__netzero__v23.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__netzero__v23.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__netzero__v23.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
     

2025-03-17 15:32:30,876 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-03-17 15:32:30,878 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [57]:
from model.report.legacy.iamc_report_hackathon import report

In [58]:
import ixmp 
mp = ixmp.Platform()

In [59]:
report(mp=mp, scen= scen2, out_dir="reporting_outputs", IDEA_format=True)


processing Table: Resource|Extraction
processing Table: Resource|Cumulative Extraction
processing Table: Primary Energy
PostProcess.out() got an unexpected keyword argument 'inpfilter'
processing Table: Primary Energy (substitution method)
PostProcess.out() got an unexpected keyword argument 'inpfilter'
processing Table: Final Energy
processing Table: Secondary Energy|Electricity
processing Table: Secondary Energy|Heat
processing Table: Secondary Energy
processing Table: Secondary Energy|Gases
processing Table: Secondary Energy|Solids
processing Table: Emissions|CO2
processing Table: Carbon Sequestration
processing Table: Emissions|BC
processing Table: Emissions|OC
processing Table: Emissions|CO
processing Table: Emissions|N2O
processing Table: Emissions|CH4
processing Table: Emissions|NH3
processing Table: Emissions|Sulfur
processing Table: Emissions|NOx
processing Table: Emissions|VOC
processing Table: Emissions|HFC
HFC125
No objects to concatenate
processing Table: Emissions
process

TypeError: cannot do slice indexing on Index with these indexers [2020] of type int

In [ ]:
caseName = netzero_1.model + '__' + netzero_1.scenario + '__v' + str(netzero_1.version)
netzero_1.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '2'})

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 03/17/25 15:18:15 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225a/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__netzero_1__v3.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__netzero_1__v3.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__netzero_1__v3.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
  

2025-03-17 15:18:43,976 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-03-17 15:18:43,979 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [ ]:
netzero_1.var('EMISS', {'node': 'Canada', 'emission': 'TCE', 'type_tec': 'all'})['lvl']

0    247.249558
1    176.027023
2    111.207788
3     72.825489
4     41.045517
5     23.917904
6     13.167351
7      6.031983
8      5.862781
Name: lvl, dtype: float64

,node,type_emission,type_tec,type_year,value,unit
0,Canada,TCE,all,cumulative,2552.472,-


(9360.0, 2552.472)